In [ ]:
%pip install dask-kubernetes

In [ ]:
from dask_kubernetes.operator import KubeCluster, make_cluster_spec
from distributed import Client

# 1. Generate the underlying default specification structure
spec = make_cluster_spec(name="my-dask-cluster", image="ghcr.io/dask/dask:latest")

# 2. Inject the mandatory "restricted" security contexts for the Scheduler Pod
spec["spec"]["scheduler"]["spec"]["securityContext"] = {
    "runAsNonRoot": True,
    "runAsUser": 1000,
    "fsGroup": 1000,
    "seccompProfile": {"type": "RuntimeDefault"}
}
spec["spec"]["scheduler"]["spec"]["containers"][0]["securityContext"] = {
    "allowPrivilegeEscalation": False,
    "capabilities": {"drop": ["ALL"]}
}

# 3. Inject the mandatory "restricted" security contexts for the Worker Pods
spec["spec"]["worker"]["spec"]["securityContext"] = {
    "runAsNonRoot": True,
    "runAsUser": 1000,
    "fsGroup": 1000,
    "seccompProfile": {"type": "RuntimeDefault"}
}
spec["spec"]["worker"]["spec"]["containers"][0]["securityContext"] = {
    "allowPrivilegeEscalation": False,
    "capabilities": {"drop": ["ALL"]}
}

# 4. Initialize the cluster using the fully complete manifest specification
cluster = KubeCluster(custom_cluster_spec=spec, namespace="c-scale-pangeo-dask")

# 5. Scale and connect
cluster.scale(2)
client = Client(cluster)

In [ ]:
client

In [ ]:
# 4. Run a quick computation to verify it works
import dask.array as da
x = da.random.random((10000, 10000), chunks=(1000, 1000))
result = x.mean().compute()

print(f"Computation successful! Result: {result}")

In [ ]:
# 5. Clean up and delete the pods
client.close()
cluster.close()